In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Helper to build summary for a telemetry dataframe
def summarize(df, file_label, name_field=None, message_freeform=False):
    # Ensure timestamp parsed to UTC datetimes
    df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    # timestamp range as unix seconds and datetimes
    min_unix = int(df['timestamp'].min()) if not df['timestamp'].empty else None
    max_unix = int(df['timestamp'].max()) if not df['timestamp'].empty else None
    min_dt = df['timestamp_dt'].min()
    max_dt = df['timestamp_dt'].max()
    # unique cmdb ids
    unique_cmdb = sorted(df['cmdb_id'].dropna().unique().tolist())
    unique_cmdb_str = ", ".join(unique_cmdb) if unique_cmdb else ""
    # unique KPI/trace/log names or freeform indicator
    if message_freeform:
        unique_names_str = "<freeform messages>"
    else:
        unique_names = sorted(df[name_field].dropna().unique().tolist()) if name_field else []
        unique_names_str = ", ".join(unique_names)
    # row counts per cmdb_id (top 20)
    counts = df.groupby('cmdb_id').size().reset_index(name='row_count').sort_values('row_count', ascending=False).head(20)
    # concise summary dataframe (one row)
    summary = pd.DataFrame([{
        'file': file_label,
        'unique_cmdb_ids': unique_cmdb_str,
        'unique_name_fields': unique_names_str,
        'total_rows': len(df),
        'min_unix_ts': min_unix,
        'max_unix_ts': max_unix,
        'min_timestamp_dt_utc': min_dt,
        'max_timestamp_dt_utc': max_dt
    }])
    return summary, counts

# Summarize each file
metric_summary, metric_counts = summarize(df_metric, "metric.csv", name_field='kpi_name')
trace_summary, trace_counts = summarize(df_trace, "trace.csv", name_field='trace_name')
log_summary, log_counts = summarize(df_log, "log.csv", name_field='log_name')
error_summary, error_counts = summarize(df_error, "error_logs.csv", name_field='message', message_freeform=True)

# Prepare heads (up to first 5 rows)
metric_head5 = df_metric.head(5)
trace_head5 = df_trace.head(5)
log_head5 = df_log.head(5)
error_head5 = df_error.head(5)

# Display summaries and compact artifacts
metric_summary, metric_counts, metric_head5, trace_summary, trace_counts, trace_head5, log_summary, log_counts, log_head5, error_summary, error_counts, error_head5

```
Out[1]:
```
```python
summary = (
    "Summary of loaded telemetry files (concise, UTC):\n\n"
    "metric.csv:\n"
    "- Unique cmdb_id (sample/list): adservice, cartservice, checkoutservice, currencyservice, "
    "emailservice, frontend, frontend-external, paymentservice, productcatalogservice, recommendationservice, redis, shippingservice.\n"
    "- Unique kpi_name (examples): cpu, diskio, error, latency-50, latency-90, mem (and others).\n"
    "- Row counts (top by cmdb_id): adservice 175, currencyservice 175, recommendationservice 175, emailservice 175, "
    "checkoutservice 150, cartservice 150, paymentservice 150, frontend 150, shippingservice 150, productcatalogservice 150, redis 98, frontend-external 50.\n"
    "- Timestamp range (unix seconds): min 1705457460, max 1705458900 (2024-01-17T02:11:00Z to 2024-01-17T02:35:00Z).\n"
    "- First row sample: timestamp 1705457460, cmdb_id adservice, kpi_name cpu, value ~4.133041e+00.\n\n"
    "trace.csv:\n"
    "- Unique cmdb_id (sample/list): checkoutservice, currencyservice, emailservice, frontendservice, productcatalogservice, recommendationservice, paymentservice, root.\n"
    "- Unique trace_name (examples): trace.from_checkoutservice.duration_mean, trace.from_checkoutservice.duration_p95, trace.from_checkoutservice.error_rate, trace.from_frontendservice.duration_mean, etc.\n"
    "- Row counts (top): checkoutservice 800, frontendservice 700, root 700, recommendationservice 500, productcatalogservice 400, currencyservice 300, emailservice 200, paymentservice 200.\n"
    "- Timestamp range: min 1705457460, max 1705458900 (2024-01-17T02:11:00Z to 2024-01-17T02:35:00Z).\n"
    "- First row sample: timestamp 1705457460, cmdb_id checkoutservice, trace_name trace.from_checkoutservice.duration_mean, value 0.004489.\n\n"
    "log.csv:\n"
    "- Unique cmdb_id (sample/list): adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, shippingservice, redis, etc.\n"
    "- Unique log_name: log.error_count, log.row_count.\n"
    "- Row counts (per cmdb_id): most services have 50 rows each (adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, shippingservice), redis has 10; total rows 460.\n"
    "- Timestamp range: min 1705457460, max 1705458900 (2024-01-17T02:11:00Z to 2024-01-17T02:35:00Z).\n"
    "- First row sample: timestamp 1705457460, cmdb_id adservice, log_name log.error_count, value 0; log.row_count sample 59.\n\n"
    "error_logs.csv:\n"
    "- Contains freeform messages (no fixed name field).\n"
    "- Total rows: 0 (empty). No timestamp range available.\n\n"
    "Overall notes:\n"
    "- All non-empty files cover the same minute-aligned time window from 2024-01-17T02:11:00Z to 2024-01-17T02:35:00Z.\n"
    "- metric.csv has 1748 rows, trace.csv 3800 rows, log.csv 460 rows, error_logs.csv is empty.\n"
    "- First-5-row samples were inspected for each file (shown previously); error_logs.csv had no rows to sample.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file                                    unique_cmdb_ids                                 unique_name_fields  total_rows  min_unix_ts  max_unix_ts      min_timestamp_dt_utc      max_timestamp_dt_utc
0  metric.csv  adservice, cartservice, checkoutservice, curre...  cpu, diskio, error, latency-50, latency-90, me...        1748   1705457460   1705458900 2024-01-17 02:11:00+00:00 2024-01-17 02:35:00+00:00,                   cmdb_id  row_count
0               adservice        175
3         currencyservice        175
9   recommendationservice        175
4            emailservice        175
2         checkoutservice        150
1             cartservice        150
7          paymentservice        150
5                frontend        150
11        shippingservice        150
8   productcatalogservice        150
10                  redis         98
6       frontend-external         50,     timestamp    cmdb_id    kpi_name         value              timestamp_dt
0  1705457460  adservice         cpu  4.133041e+00 2024-01-17 02:11:00+00:00
1  1705457460  adservice      diskio  7.596165e+04 2024-01-17 02:11:00+00:00
2  1705457460  adservice  latency-50  3.246206e-03 2024-01-17 02:11:00+00:00
3  1705457460  adservice  latency-90  5.842105e-03 2024-01-17 02:11:00+00:00
4  1705457460  adservice         mem  2.081331e+08 2024-01-17 02:11:00+00:00,         file                                    unique_cmdb_ids                                 unique_name_fields  total_rows  min_unix_ts  max_unix_ts      min_timestamp_dt_utc      max_timestamp_dt_utc
0  trace.csv  checkoutservice, currencyservice, emailservice...  trace.from_checkoutservice.duration_mean, trac...        3800   1705457460   1705458900 2024-01-17 02:11:00+00:00 2024-01-17 02:35:00+00:00,                  cmdb_id  row_count
0        checkoutservice        800
3        frontendservice        700
7                   root        700
6  recommendationservice        500
5  productcatalogservice        400
1        currencyservice        300
2           emailservice        200
4         paymentservice        200,     timestamp          cmdb_id                                trace_name      value              timestamp_dt
0  1705457460  checkoutservice  trace.from_checkoutservice.duration_mean   0.004489 2024-01-17 02:11:00+00:00
1  1705457460  checkoutservice   trace.from_checkoutservice.duration_p95   0.007249 2024-01-17 02:11:00+00:00
2  1705457460  checkoutservice     trace.from_checkoutservice.error_rate   0.000000 2024-01-17 02:11:00+00:00
3  1705457460  checkoutservice      trace.from_checkoutservice.row_count  22.000000 2024-01-17 02:11:00+00:00
4  1705457460  checkoutservice  trace.from_frontendservice.duration_mean   0.110681 2024-01-17 02:11:00+00:00,       file                                    unique_cmdb_ids              unique_name_fields  total_rows  min_unix_ts  max_unix_ts      min_timestamp_dt_utc      max_timestamp_dt_utc
0  log.csv  adservice, cartservice, checkoutservice, curre...  log.error_count, log.row_count         460   1705457460   1705458900 2024-01-17 02:11:00+00:00 2024-01-17 02:35:00+00:00,                  cmdb_id  row_count
0              adservice         50
1            cartservice         50
2        checkoutservice         50
3        currencyservice         50
4           emailservice         50
5               frontend         50
6         paymentservice         50
7  recommendationservice         50
9        shippingservice         50
8                  redis         10,     timestamp          cmdb_id         log_name  value              timestamp_dt
0  1705457460        adservice  log.error_count      0 2024-01-17 02:11:00+00:00
1  1705457460        adservice    log.row_count     59 2024-01-17 02:11:00+00:00
2  1705457460      cartservice  log.error_count      0 2024-01-17 02:11:00+00:00
3  1705457460      cartservice    log.row_count    105 2024-01-17 02:11:00+00:00
4  1705457460  checkoutservice  log.error_count      0 2024-01-17 02:11:00+00:00,              file unique_cmdb_ids   unique_name_fields  total_rows min_unix_ts max_unix_ts min_timestamp_dt_utc max_timestamp_dt_utc
0  error_logs.csv                  <freeform messages>           0        None        None                  NaT                  NaT, Empty DataFrame
Columns: [cmdb_id, row_count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message, error, timestamp_dt]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse previously loaded df_metric; ensure timestamp_dt exists and is parsed in UTC
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Services of interest
services = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# 1) Compute global P95 threshold per (cmdb_id, kpi_name) using the full series (no time filtering)
p95_all = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# Keep only the five services combos for reporting later
p95_selected = p95_all[p95_all['cmdb_id'].isin(services)].copy()

# 2) Filter metric points to the incident window (UTC)
window_start = pd.Timestamp("2024-01-17 02:08:52", tz='UTC')
window_end = pd.Timestamp("2024-01-17 02:38:52", tz='UTC')

df_window = df_metric[
    (df_metric['cmdb_id'].isin(services)) &
    (df_metric['timestamp_dt'] >= window_start) &
    (df_metric['timestamp_dt'] <= window_end)
].copy()

# 3) Join windowed points with precomputed p95 thresholds
df_win_with_p95 = df_window.merge(p95_selected, on=['cmdb_id', 'kpi_name'], how='left')

# Identify anomalies: value >= p95_threshold (ensure p95 exists; if no p95, treat as no anomaly)
df_win_with_p95['is_anomaly'] = False
mask_has_p95 = df_win_with_p95['p95_threshold'].notnull()
df_win_with_p95.loc[mask_has_p95, 'is_anomaly'] = df_win_with_p95.loc[mask_has_p95, 'value'] >= df_win_with_p95.loc[mask_has_p95, 'p95_threshold']

# Aggregate per pair
agg_anoms = (
    df_win_with_p95
    .groupby(['cmdb_id', 'kpi_name'], observed=True)
    .agg(
        p95_threshold=('p95_threshold', 'first'),
        total_points_in_window=('value', 'size'),
        count_anomalies_in_window=('is_anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# Earliest anomaly timestamp (unix seconds) per pair
anomaly_times = (
    df_win_with_p95[df_win_with_p95['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['timestamp']
    .min()
    .reset_index(name='earliest_anomaly_timestamp_unix')
)

# Merge earliest times
agg_anoms = agg_anoms.merge(anomaly_times, on=['cmdb_id','kpi_name'], how='left')

# Compute max_over_threshold_ratio where anomalies exist
agg_anoms['max_over_threshold_ratio'] = np.where(
    agg_anoms['count_anomalies_in_window'] > 0,
    np.where(agg_anoms['p95_threshold'] != 0,
             (agg_anoms['max_value_in_window'] - agg_anoms['p95_threshold']) / agg_anoms['p95_threshold'],
             np.nan),
    np.nan
)

# Keep only rows where count_anomalies_in_window > 0
result = (
    agg_anoms[agg_anoms['count_anomalies_in_window'] > 0]
    .sort_values('count_anomalies_in_window', ascending=False)
    .reset_index(drop=True)
)

# Limit to top 20 rows and select compact columns
result_compact = result.loc[:, [
    'cmdb_id',
    'kpi_name',
    'p95_threshold',
    'count_anomalies_in_window',
    'earliest_anomaly_timestamp_unix',
    'max_value_in_window',
    'max_over_threshold_ratio'
]].head(20)

# Also prepare the short list of all (cmdb_id, unique kpi_name) combinations discovered for these five services
combos = (
    df_metric[df_metric['cmdb_id'].isin(services)]
    [['cmdb_id','kpi_name']]
    .drop_duplicates()
    .sort_values(['cmdb_id','kpi_name'])
    .reset_index(drop=True)
)

# Display compact outputs: (1) anomalies summary table, (2) combos list (compact)
result_compact, combos.head(200)

```
Out[2]:
```
```python
summary = (
    "Summary of metric anomaly scan for services [checkoutservice, currencyservice, emailservice, "
    "productcatalogservice, recommendationservice] within 2024-01-17 02:08:52–02:38:52 UTC:\n\n"
    "1) Combos discovered: 33 unique (cmdb_id, kpi_name) combinations across the five services.\n\n"
    "2) Top anomaly findings (rows where count_anomalies_in_window > 0, sorted by count):\n"
    "- currencyservice / error: p95=0.0, anomalies=25, earliest=1705457460, max=0.0 (p95==0 -> ratio N/A)\n"
    "- recommendationservice / diskio: p95=0.0, anomalies=25, earliest=1705457460, max=0.0 (p95==0 -> ratio N/A)\n"
    "- emailservice / diskio: p95=0.0, anomalies=25, earliest=1705457460, max=0.0 (p95==0 -> ratio N/A)\n\n"
    "3) Other detected anomaly pairs (each with count_anomalies_in_window = 2):\n"
    "- checkoutservice: cpu, mem, socket, workload, latency-50, latency-90\n"
    "- currencyservice: cpu, latency-50, latency-90, socket, mem, workload\n"
    "- emailservice: cpu, latency-50, latency-90, mem, workload\n\n"
    "Notes on severity and timing:\n"
    "- Many anomalies are small in count (mostly 2 occurrences) and have modest over-threshold ratios; "
    "max_over_threshold_ratio values for non-zero p95 cases range roughly from ~0.01 up to ~0.46 (largest on mem metrics).\n"
    "- Earliest anomalies mostly occur at unix ts 1705457460 (2024-01-17 02:11:00 UTC) or nearby minutes.\n\n"
    "Interpretation / next steps:\n"
    "- The highest raw anomaly counts are for three metrics with p95==0 (error and diskio entries). A p95 of zero indicates the global series is often zero; treat these carefully (zero thresholds produce NaN ratio). Investigate whether the metric is sparse or mis-normalized.\n"
    "- Memory (mem) and socket metrics for checkoutservice and emailservice show the largest relative excursions (ratios up to ~0.44–0.46) and should be prioritized for follow-up.\n"
    "- Recommend drilling into raw series and logs around the earliest anomaly times (e.g., 2024-01-17T02:11:00Z) for the listed pairs to determine root cause.\n\n"
    "If you want the exact compact table or the full list of the 33 (cmdb_id,kpi_name) combos shown as text, I can return it next."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  p95_threshold  count_anomalies_in_window  earliest_anomaly_timestamp_unix  max_value_in_window  max_over_threshold_ratio
0         currencyservice       error   0.000000e+00                         25                       1705457460         0.000000e+00                       NaN
1   recommendationservice      diskio   0.000000e+00                         25                       1705457460         0.000000e+00                       NaN
2            emailservice      diskio   0.000000e+00                         25                       1705457460         0.000000e+00                       NaN
3         checkoutservice         cpu   5.211649e-01                          2                       1705457760         5.434223e-01                  0.042707
4         checkoutservice         mem   1.567353e+07                          2                       1705457460         2.262016e+07                  0.443208
5         checkoutservice      socket   1.360000e+01                          2                       1705457460         1.900000e+01                  0.397059
6         checkoutservice    workload   1.107277e+00                          2                       1705457460         1.315417e+00                  0.187975
7         currencyservice         cpu   3.033509e+01                          2                       1705457640         3.136441e+01                  0.033931
8         checkoutservice  latency-50   1.666335e-01                          2                       1705458540         1.695026e-01                  0.017218
9         currencyservice  latency-50   4.660943e-03                          2                       1705457460         4.712024e-03                  0.010959
10        currencyservice  latency-90   7.358266e-02                          2                       1705457520         7.373475e-02                  0.002067
11        currencyservice      socket   6.226667e+00                          2                       1705457460         7.000000e+00                  0.124197
12        currencyservice         mem   1.772846e+08                          2                       1705457460         2.102436e+08                  0.185910
13        currencyservice    workload   8.362886e+01                          2                       1705457700         8.469668e+01                  0.012769
14           emailservice         cpu   3.040380e-01                          2                       1705457460         3.259599e-01                  0.072103
15           emailservice  latency-50   3.068822e-03                          2                       1705458300         3.165970e-03                  0.031656
16        checkoutservice  latency-90   2.702459e-01                          2                       1705458120         3.156881e-01                  0.168151
17           emailservice  latency-90   4.723879e-03                          2                       1705458300         4.959012e-03                  0.049775
18           emailservice         mem   5.951261e+07                          2                       1705457460         8.694989e+07                  0.461033
19           emailservice    workload   1.113947e+00                          2                       1705457940         1.312350e+00                  0.178108,                   cmdb_id    kpi_name
0         checkoutservice         cpu
1         checkoutservice  latency-50
2         checkoutservice  latency-90
3         checkoutservice         mem
4         checkoutservice      socket
..                    ...         ...
28  recommendationservice  latency-50
29  recommendationservice  latency-90
30  recommendationservice         mem
31  recommendationservice      socket
32  recommendationservice    workload

[33 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse df_trace loaded earlier; ensure timestamp_dt exists and is parsed in UTC
if 'df_trace' not in globals():
    df_trace = pd.read_csv("trace.csv")
if 'timestamp_dt' not in df_trace.columns:
    df_trace['timestamp_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Services of interest
services = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# 1) Compute global P95 threshold per (cmdb_id, trace_name) using the full series (no time filtering)
p95_all_trace = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# Keep only the five services
p95_selected = p95_all_trace[p95_all_trace['cmdb_id'].isin(services)].copy()

# 2) Filter trace points to the incident window (UTC)
window_start = pd.Timestamp("2024-01-17 02:08:52", tz='UTC')
window_end = pd.Timestamp("2024-01-17 02:38:52", tz='UTC')

df_trace_window = df_trace[
    (df_trace['cmdb_id'].isin(services)) &
    (df_trace['timestamp_dt'] >= window_start) &
    (df_trace['timestamp_dt'] <= window_end)
].copy()

# 3) Merge windowed points with precomputed p95 thresholds
df_twin_with_p95 = df_trace_window.merge(p95_selected, on=['cmdb_id', 'trace_name'], how='left')

# Identify anomalies: value >= p95_threshold (only where p95 exists)
df_twin_with_p95['is_anomaly'] = False
mask_has_p95 = df_twin_with_p95['p95_threshold'].notnull()
df_twin_with_p95.loc[mask_has_p95, 'is_anomaly'] = df_twin_with_p95.loc[mask_has_p95, 'value'] >= df_twin_with_p95.loc[mask_has_p95, 'p95_threshold']

# Aggregate per pair
agg_trace = (
    df_twin_with_p95
    .groupby(['cmdb_id', 'trace_name'], observed=True)
    .agg(
        p95_threshold=('p95_threshold', 'first'),
        total_points_in_window=('value', 'size'),
        count_anomalies_in_window=('is_anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# Earliest anomaly timestamp (unix seconds) per pair
anomaly_times_trace = (
    df_twin_with_p95[df_twin_with_p95['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'], observed=True)['timestamp']
    .min()
    .reset_index(name='earliest_anomaly_timestamp_unix')
)

# Merge earliest times
agg_trace = agg_trace.merge(anomaly_times_trace, on=['cmdb_id','trace_name'], how='left')

# Compute max_over_threshold_ratio where p95 != 0 and anomalies exist; else NaN
agg_trace['max_over_threshold_ratio'] = np.where(
    agg_trace['count_anomalies_in_window'] > 0,
    np.where(agg_trace['p95_threshold'] != 0,
             (agg_trace['max_value_in_window'] - agg_trace['p95_threshold']) / agg_trace['p95_threshold'],
             np.nan),
    np.nan
)

# Filter to only rows with anomalies and sort by count desc, limit top 20
trace_anomalies = (
    agg_trace[agg_trace['count_anomalies_in_window'] > 0]
    .sort_values('count_anomalies_in_window', ascending=False)
    .reset_index(drop=True)
)

# Compact result table with requested columns, top 20
result_trace_compact = trace_anomalies.loc[:, [
    'cmdb_id',
    'trace_name',
    'p95_threshold',
    'count_anomalies_in_window',
    'earliest_anomaly_timestamp_unix',
    'max_value_in_window',
    'max_over_threshold_ratio'
]].head(20)

# Extract caller services from anomaly rows' trace_name patterns: substring after 'trace.from_' and before next '.'
anomaly_trace_names = result_trace_compact['trace_name'].dropna().astype(str)
callers = anomaly_trace_names.str.extract(r'trace\.from_([^\.]+)', expand=False).dropna().unique()
callers_df = pd.DataFrame({'caller_service': sorted(callers)})

# Display compact outputs: (1) anomaly summary table (top 20), (2) deduplicated caller services observed among anomaly rows
result_trace_compact, callers_df

```
Out[3]:
```
```python
summary = (
    "Trace anomaly summary for services [checkoutservice, currencyservice, emailservice, "
    "productcatalogservice, recommendationservice] within 2024-01-17 02:08:52–02:38:52 UTC:\n\n"
    "- Findings: All reported anomaly rows are for trace error_rate metrics (trace.*.error_rate).\n"
    "- Counts: Each listed (cmdb_id, trace_name) pair shows count_anomalies_in_window = 25 and p95_threshold = 0.0.\n"
    "- Timing: Earliest anomaly timestamp for these rows is unix 1705457460 (2024-01-17 02:11:00 UTC).\n"
    "- Values: max_value_in_window is 0.0 for the anomaly rows, so max_over_threshold_ratio is not defined (p95==0).\n"
    "- Caller services (parsed from trace_name in anomaly rows): checkoutservice, frontendservice, recommendationservice, root.\n\n"
    "Interpretation and next steps:\n"
    "- A global P95 of 0.0 for error_rate and zero max values in-window indicates the error_rate metric is sparse/zero-valued "
    "and the numeric thresholds produce ambiguous 'anomalies'. Treat these results cautiously (p95==0 yields NaN ratios).\n"
    "- The uniform earliest timestamp (2024-01-17T02:11:00Z) suggests simultaneous zero-valued/error_rate entries across many trace types at that minute.\n"
    "- Recommended follow-up: inspect raw trace payloads and service logs around 2024-01-17T02:11:00Z for the listed services and callers to determine whether this represents real errors, missing instrumentation, or a normalization artifact."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                   trace_name  p95_threshold  count_anomalies_in_window  earliest_anomaly_timestamp_unix  max_value_in_window  max_over_threshold_ratio
0         checkoutservice        trace.from_checkoutservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
1         checkoutservice        trace.from_frontendservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
2         checkoutservice          trace.to_checkoutservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
3         checkoutservice                   trace.from_root.error_rate            0.0                         25                       1705457460                  0.0                       NaN
4         checkoutservice          trace.to_currencyservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
5         checkoutservice             trace.to_emailservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
6         checkoutservice    trace.to_productcatalogservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
7         checkoutservice           trace.to_paymentservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
8   productcatalogservice        trace.from_frontendservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
9   productcatalogservice  trace.from_recommendationservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
10  recommendationservice        trace.from_frontendservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
11  productcatalogservice                   trace.from_root.error_rate            0.0                         25                       1705457460                  0.0                       NaN
12  recommendationservice    trace.to_productcatalogservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
13  recommendationservice    trace.to_recommendationservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
14  recommendationservice                   trace.from_root.error_rate            0.0                         25                       1705457460                  0.0                       NaN
15  recommendationservice  trace.from_recommendationservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
16        currencyservice                   trace.from_root.error_rate            0.0                         25                       1705457460                  0.0                       NaN
17           emailservice        trace.from_checkoutservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
18  productcatalogservice        trace.from_checkoutservice.error_rate            0.0                         25                       1705457460                  0.0                       NaN
19           emailservice                   trace.from_root.error_rate            0.0                         25                       1705457460                  0.0                       NaN,           caller_service
0        checkoutservice
1        frontendservice
2  recommendationservice
3                   root)```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd

# Reuse df_log from earlier; ensure timestamp_dt is present and parsed in UTC
if 'df_log' not in globals():
    df_log = pd.read_csv("log.csv")
if 'timestamp_dt' not in df_log.columns:
    df_log['timestamp_dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Services of interest (reuse if available)
services = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# 1) Compute global P95 per (cmdb_id, log_name) using the full series
p95_all_log = (
    df_log
    .groupby(['cmdb_id', 'log_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# Keep only the five services
p95_selected = p95_all_log[p95_all_log['cmdb_id'].isin(services)].copy()

# 2) Filter log points to the incident window (UTC)
window_start = pd.Timestamp("2024-01-17 02:08:52", tz='UTC')
window_end = pd.Timestamp("2024-01-17 02:38:52", tz='UTC')

df_log_window = df_log[
    (df_log['cmdb_id'].isin(services)) &
    (df_log['timestamp_dt'] >= window_start) &
    (df_log['timestamp_dt'] <= window_end)
].copy()

# 3) Merge windowed points with precomputed p95 thresholds
df_lwin_with_p95 = df_log_window.merge(p95_selected, on=['cmdb_id', 'log_name'], how='left')

# Identify anomalies: value >= p95_threshold (only where p95 exists)
df_lwin_with_p95['is_anomaly'] = False
mask_has_p95 = df_lwin_with_p95['p95_threshold'].notnull()
df_lwin_with_p95.loc[mask_has_p95, 'is_anomaly'] = df_lwin_with_p95.loc[mask_has_p95, 'value'] >= df_lwin_with_p95.loc[mask_has_p95, 'p95_threshold']

# Aggregate per pair
agg_log = (
    df_lwin_with_p95
    .groupby(['cmdb_id', 'log_name'], observed=True)
    .agg(
        p95_threshold=('p95_threshold', 'first'),
        count_anomalies_in_window=('is_anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# Earliest anomaly timestamp (unix seconds) per pair
anomaly_times_log = (
    df_lwin_with_p95[df_lwin_with_p95['is_anomaly']]
    .groupby(['cmdb_id', 'log_name'], observed=True)['timestamp']
    .min()
    .reset_index(name='earliest_anomaly_timestamp_unix')
)

# Merge earliest times
agg_log = agg_log.merge(anomaly_times_log, on=['cmdb_id','log_name'], how='left')

# Compute max_over_threshold_ratio where p95 != 0 and anomalies exist; else NaN
agg_log['max_over_threshold_ratio'] = np.where(
    agg_log['count_anomalies_in_window'] > 0,
    np.where(agg_log['p95_threshold'] != 0,
             (agg_log['max_value_in_window'] - agg_log['p95_threshold']) / agg_log['p95_threshold'],
             np.nan),
    np.nan
)

# Filter to only rows with anomalies and sort by count desc, limit top 20
log_anomalies = (
    agg_log[agg_log['count_anomalies_in_window'] > 0]
    .sort_values('count_anomalies_in_window', ascending=False)
    .reset_index(drop=True)
)

# Compact result table with requested columns, top 20
result_log_compact = log_anomalies.loc[:, [
    'cmdb_id',
    'log_name',
    'p95_threshold',
    'count_anomalies_in_window',
    'earliest_anomaly_timestamp_unix',
    'max_value_in_window',
    'max_over_threshold_ratio'
]].head(20)

# Identify top two cmdb_id by total count_anomalies_in_window
top_cmdb_counts = (
    log_anomalies
    .groupby('cmdb_id', observed=True)['count_anomalies_in_window']
    .sum()
    .reset_index(name='total_anomalies')
    .sort_values('total_anomalies', ascending=False)
    .reset_index(drop=True)
)

top_two = top_cmdb_counts.head(2)

# For each of those top two services, get up to 10 raw log rows in the incident window sorted by timestamp asc
top_rows = {}
for cmdb in top_two['cmdb_id'].tolist():
    rows = (
        df_log_window[df_log_window['cmdb_id'] == cmdb]
        .sort_values('timestamp_dt', ascending=True)
        [['timestamp','cmdb_id','log_name','value']]
        .head(10)
        .reset_index(drop=True)
    )
    top_rows[cmdb] = rows

# Display compact outputs:
# (1) result_log_compact: anomaly table (top 20)
# (2) top_two: top two cmdb_id by total anomalies
# (3) top_rows: dict mapping cmdb_id to up-to-10 raw rows DataFrame
result_log_compact, top_two, top_rows

```
Out[4]:
```
Summary of log.csv analysis for services [checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice] (incident window 2024-01-17 02:08:52–02:38:52 UTC):

Key anomaly rows (only pairs with count_anomalies_in_window > 0)
- checkoutservice / log.error_count — p95 = 0.0, anomalies = 25, earliest = 1705457460 (2024-01-17 02:11:00 UTC), max = 0 (ratio N/A because p95 == 0)
- currencyservice / log.error_count — p95 = 0.0, anomalies = 25, earliest = 1705457460, max = 0 (ratio N/A)
- recommendationservice / log.error_count — p95 = 0.0, anomalies = 25, earliest = 1705457460, max = 0 (ratio N/A)
- emailservice / log.error_count — p95 = 0.0, anomalies = 25, earliest = 1705457460, max = 0 (ratio N/A)
- currencyservice / log.row_count — p95 = 2522.0, anomalies = 2, earliest = 1705457700 (2024-01-17 02:15:00 UTC), max = 2625, max_over_threshold_ratio ≈ 0.04084
- checkoutservice / log.row_count — p95 = 105.6, anomalies = 2, earliest = 1705457700, max = 114, max_over_threshold_ratio ≈ 0.07955
- emailservice / log.row_count — p95 = 35.2, anomalies = 2, earliest = 1705457700, max = 38, max_over_threshold_ratio ≈ 0.07955
- recommendationservice / log.row_count — p95 = 575.4, anomalies = 2, earliest = 1705457700, max = 587, max_over_threshold_ratio ≈ 0.02016

Top services by total anomaly counts
- checkoutservice: total_anomalies = 27
- currencyservice: total_anomalies = 27

Up to 10 raw log rows (timestamp, cmdb_id, log_name, value) in the incident window for the top two services (sorted by timestamp ascending)

checkoutservice:
- 1705457460, checkoutservice, log.error_count, 0
- 1705457460, checkoutservice, log.row_count, 11
- 1705457520, checkoutservice, log.error_count, 0
- 1705457520, checkoutservice, log.row_count, 63
- 1705457580, checkoutservice, log.error_count, 0
- 1705457580, checkoutservice, log.row_count, 90
- 1705457640, checkoutservice, log.error_count, 0
- 1705457640, checkoutservice, log.row_count, 57
- 1705457700, checkoutservice, log.error_count, 0
- 1705457700, checkoutservice, log.row_count, 108

currencyservice:
- 1705457460, currencyservice, log.error_count, 0
- 1705457460, currencyservice, log.row_count, 359
- 1705457520, currencyservice, log.error_count, 0
- 1705457520, currencyservice, log.row_count, 2197
- 1705457580, currencyservice, log.error_count, 0
- 1705457580, currencyservice, log.row_count, 2311
- 1705457640, currencyservice, log.error_count, 0
- 1705457640, currencyservice, log.row_count, 2409
- 1705457700, currencyservice, log.error_count, 0
- 1705457700, currencyservice, log.row_count, 2524

Interpretation and recommended next steps
- Many log.error_count anomalies have p95 == 0 and max == 0. This indicates the error_count series is typically zero and the p95 = 0 makes the anomaly flag ambiguous — treat these counts cautiously (they may reflect sparse metrics or normalization artifacts rather than real error spikes).
- The log.row_count anomalies (small counts = 2 each) show real positive increases around unix 1705457700 (2024-01-17 02:15:00 UTC) for checkoutservice, currencyservice, emailservice, and recommendationservice — these are modest but measurable increases above their global P95 values.
- Actionable follow-up: inspect raw logs and application traces around 02:11:00 UTC (earliest anomaly minute) and 02:15:00 UTC (row_count spikes) for checkoutservice and currencyservice to determine whether these represent actual issues (increased traffic, log volume changes, or instrumentation artifacts).

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id         log_name  p95_threshold  count_anomalies_in_window  earliest_anomaly_timestamp_unix  max_value_in_window  max_over_threshold_ratio
0        checkoutservice  log.error_count            0.0                         25                       1705457460                    0                       NaN
1        currencyservice  log.error_count            0.0                         25                       1705457460                    0                       NaN
2  recommendationservice  log.error_count            0.0                         25                       1705457460                    0                       NaN
3           emailservice  log.error_count            0.0                         25                       1705457460                    0                       NaN
4        currencyservice    log.row_count         2522.0                          2                       1705457700                 2625                  0.040841
5        checkoutservice    log.row_count          105.6                          2                       1705457700                  114                  0.079545
6           emailservice    log.row_count           35.2                          2                       1705457700                   38                  0.079545
7  recommendationservice    log.row_count          575.4                          2                       1705457700                  587                  0.020160,            cmdb_id  total_anomalies
0  checkoutservice               27
1  currencyservice               27, {'checkoutservice':     timestamp          cmdb_id         log_name  value
0  1705457460  checkoutservice  log.error_count      0
1  1705457460  checkoutservice    log.row_count     11
2  1705457520  checkoutservice  log.error_count      0
3  1705457520  checkoutservice    log.row_count     63
4  1705457580  checkoutservice  log.error_count      0
5  1705457580  checkoutservice    log.row_count     90
6  1705457640  checkoutservice  log.error_count      0
7  1705457640  checkoutservice    log.row_count     57
8  1705457700  checkoutservice  log.error_count      0
9  1705457700  checkoutservice    log.row_count    108, 'currencyservice':     timestamp          cmdb_id         log_name  value
0  1705457460  currencyservice  log.error_count      0
1  1705457460  currencyservice    log.row_count    359
2  1705457520  currencyservice  log.error_count      0
3  1705457520  currencyservice    log.row_count   2197
4  1705457580  currencyservice  log.error_count      0
5  1705457580  currencyservice    log.row_count   2311
6  1705457640  currencyservice  log.error_count      0
7  1705457640  currencyservice    log.row_count   2409
8  1705457700  currencyservice  log.error_count      0
9  1705457700  currencyservice    log.row_count   2524})```
```